# Four matched phone episodes: independent CPU grading81
Same frozen model/APK, two exposed cases and two counterbalanced policy arms. Fresh original clean/export/reference checks; absent or failed verifiers remain unknown. No GPU/model, no holdout or prize claim.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
SESSION_STARTED_MONOTONIC = time.monotonic()
MODE = 'cpu_controls'
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not IS_KAGGLE:
    print('Definitions only: local execution performs no official inference, fixture grading or installation')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
PLAN = json.loads('{"tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "guide_sha256": "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb", "expected_core_packages": {"adk-eval-core": "0.1.0", "adk-submission": "0.2.13", "swegemma": "0.2.10"}, "expected_verify_task_sha256": "dc89c3595217c5627d73dcccf41b90a45a2743f2f770537e2f5d2da0b3bced16", "expected_organizer_python_sources": [{"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/__init__.py", "bytes": 4122, "sha256": "528e02f35eaef6fa733acc0a026c80cc191d28d4a4192a740f9b7bc8a88c9211"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/__init__.py", "bytes": 254, "sha256": "4a709384174fb5be5aa22c9a1385ba830d82bf03fab1798280b350311e75588e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/budget.py", "bytes": 15780, "sha256": "16de25ae326f0f8440772672a4bf7198a4ebb5586c7c3f2b31048ca896f04ab6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/__init__.py", "bytes": 266, "sha256": "6e4f2c12837a366147d582b9ccfc4f6c3fd0d117e311767e292abe8b84642397"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/edit.py", "bytes": 14676, "sha256": "a78a761d5078df92606c63b3adb61991a2878bcbb80204875d241175cb532a6f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/errors.py", "bytes": 5691, "sha256": "2a8d545695a89c0abbc8984d771df6ae31155998b8e20cb91da02d0a5b900373"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/__init__.py", "bytes": 246, "sha256": "1a9e1673840b8be5bcbf93767a1d130c007abf350cfc2c263e389135bba223af"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/base.py", "bytes": 9724, "sha256": "bdedef267b1022ffac970c7bc18c2d0a53753ce1f120a3c355b91c9b7b666a0d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/__init__.py", "bytes": 227, "sha256": "dd53fad958a5719da6d6ab6869f59daa7c70060a257dedc77551873557b829f9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/display_plugin.py", "bytes": 5648, "sha256": "7250793fa30985086c193fc55d5b81f4c4d115b44ad764e546a8cc1b0ac20d69"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/retry_plugin.py", "bytes": 7084, "sha256": "ba61cf201167aaf970c3131a4f24b464515509f3cacd5358de99c5ec0fb07af0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/__init__.py", "bytes": 293, "sha256": "2faaa7d8445299656396987f7b55876025b5dfaab793afa0c11027ab08984b01"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/base.py", "bytes": 1198, "sha256": "fa97acbfdf8be95b9b247e525fb4fa682c8b8da156b405f5bc29f684eafb4d5f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/runner.py", "bytes": 3321, "sha256": "0c057f402eb42f0bdc4e84fd88d064287f3929284d8f345813df8dedeb7b9cc3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/__init__.py", "bytes": 751, "sha256": "b9f13699e2bc09fa57815bb3c6bf07c4aefb9ab5b74e80ec46625f8421b10d5a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/base.py", "bytes": 7694, "sha256": "da53e3bb0dfb7d2490d63195740d94560291c37001c4ebae4d12733cf4856007"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/containers.py", "bytes": 21219, "sha256": "c36a7077901aafcd8a16ae22441dfe40d8713e36bbc8534ab2bad1cd953bf769"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/docker_sandbox.py", "bytes": 7011, "sha256": "484b8864c2ff876f10b85f95e69e72b245e8ca5f6f30dc1020baec846212c899"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/subprocess_sandbox.py", "bytes": 11715, "sha256": "09dc213bd71f4ad33a2a403732271efbc78bd09db2d2999817d55f4e630ac6fc"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/__init__.py", "bytes": 408, "sha256": "d1b4ed94c3c5fa97541f593d62645cc73fe60587da6d2deb742c48ac00f01096"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif.py", "bytes": 5302, "sha256": "266324e80f6a4fc23aa3d809f677a6e950a7bc971f8e903ce3f55dc2c0cd8f35"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif_converter.py", "bytes": 7430, "sha256": "1d35490d305e1676d09c11a54301488cdecc6aceb03ee3469831d3f8a1cdd992"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/markdown.py", "bytes": 7160, "sha256": "e8e07b6a71926196e09a18f08f0cc3ff6da5820bff95fcc0534fdeab083fbb6d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/trace.py", "bytes": 23186, "sha256": "b426e17c03b420f38ee4e08c038678ff4ba66d07ea893638f4e22a9fba8a4bc5"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/__init__.py", "bytes": 1571, "sha256": "3f39053c2fd360dedca776c611df55b0a4382e910f3681b0f9dbb3f08b0b459d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/dashboard.py", "bytes": 18158, "sha256": "c56edca53fe06efb29151ef140b25ac99f795b08faf0bf0632abae18ce544eec"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/event_display.py", "bytes": 23833, "sha256": "4efbb0a55b5cd93cd0d8036c2d3f5f65b67072a0f5dc197e2e7b360a791f4a3e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/__init__.py", "bytes": 1170, "sha256": "2502098e77ae11a186177c980cb30560e3cd90703026a0c3dda95e191bbce288"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/command.py", "bytes": 4046, "sha256": "50856b977eb86fd21f312a340e6d782101b934564a3527348e869294aa84d960"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/file.py", "bytes": 5709, "sha256": "5ba0b3a5e51925431545a0f2f408725d4e2e487b934f821c635fb364b10511bf"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/generic.py", "bytes": 3922, "sha256": "f458b8993b7158c3a5a73991c0adb3cb35509cd927a8ab3b14df4a515020b3be"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/models.py", "bytes": 5146, "sha256": "5445d681fa4f4dc9847cda9388af8c623fd7e1e73bce15f8c42de9e34a8f84ea"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/python.py", "bytes": 2119, "sha256": "3249c42b0b2d73a447913d07076623f3dbd7401b63726d9e3a3a9777ff5906e0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/registry.py", "bytes": 6171, "sha256": "b63f7260e8219d36a3c3f4ebd1001c7d729d1f106446b9ca50620f79a2e092c6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/hud.py", "bytes": 14051, "sha256": "bfc95127fd9594e5e9269071f3d7241875ab33d026316482cb157111a790757c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/presenter.py", "bytes": 7238, "sha256": "626d70427727c5e0a8ffcf37032323c3516461640694fd9be2968dbabb8a96d4"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/protocol.py", "bytes": 4050, "sha256": "84d85f3e5adbc74feb7d4bedaf96131541cad5ac57012abb86f9cf4b046feb8a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/summary.py", "bytes": 592, "sha256": "3bba083cf4d57081c50abb1af49307abe586a39ea1ebf2b9891783331085b2fe"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/__init__.py", "bytes": 437, "sha256": "630451edfc5b0032078ed858d971e8520f2ea776a2231aa5b7dc1a9a1958cd78"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/scoring.py", "bytes": 3505, "sha256": "f6c561e79fbfea17be26c526e96d0ed11a6d57edada61ff2cd512f29eceeb69d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/utils.py", "bytes": 4562, "sha256": "df512c616a387afdb69eb5841ad8aa7915f042f9c85ab17594f03bf4a27f2df3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/__init__.py", "bytes": 159, "sha256": "e477b6d54612929ba547af4d829ece61a74882c7d2d92c29f76f2e97c370bcf9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/templates.py", "bytes": 9661, "sha256": "d7ec71cf42a84cb1c196ad9e7b4bef95611f6353a7d22182b7b5f147d9581b03"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/viewer.py", "bytes": 27504, "sha256": "16e4eebf920d0de53587c4008d1a0c4b365527940aef2f278c0cfc5fc3c14fc9"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/__init__.py", "bytes": 4369, "sha256": "4b89347769780899d22628508e90e1066e5824e586bd1da60c4f817dc91f1b70"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/__init__.py", "bytes": 637, "sha256": "ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/llm.py", "bytes": 3658, "sha256": "97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/sub_agents.py", "bytes": 2801, "sha256": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/workflows.py", "bytes": 5565, "sha256": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/compiler.py", "bytes": 10492, "sha256": "39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/context.py", "bytes": 7403, "sha256": "3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/discovery.py", "bytes": 31163, "sha256": "ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/errors.py", "bytes": 13999, "sha256": "7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/limits.py", "bytes": 18965, "sha256": "afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/paths.py", "bytes": 9379, "sha256": "7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/registry.py", "bytes": 11173, "sha256": "c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/__init__.py", "bytes": 800, "sha256": "b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/callbacks.py", "bytes": 3643, "sha256": "332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/generation.py", "bytes": 18037, "sha256": "ba0d58d1dce4395d656416522855cab07d43b7ed0cd61898234dc35e9b92ecc6"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/models.py", "bytes": 3179, "sha256": "39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/tools.py", "bytes": 10593, "sha256": "ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/schema.py", "bytes": 17404, "sha256": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/server.py", "bytes": 35447, "sha256": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/yaml_loader.py", "bytes": 12618, "sha256": "01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/__init__.py", "bytes": 3858, "sha256": "9d65aa619c1df4895d952a8b58aa9fa2dfb0e0d75edf8c5f41c9bc1a0cc06bf1"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/__main__.py", "bytes": 145, "sha256": "5cd5c92ee1a230e24c17d693c3093edf0e3e78b96c08815351ba82432a1aa4f4"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/budget.py", "bytes": 1305, "sha256": "60bd0f73e0fcede68a3024a5e37836b0bcb0d133952044d7dc3e5a4102f2c2f9"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/cli.py", "bytes": 5221, "sha256": "7197c5fd6b141d7df805dd350130944c84b0b1fc14ed5eeef3f52b2e26d5e30d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/config.py", "bytes": 7440, "sha256": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/context.py", "bytes": 23710, "sha256": "97f06280c50df3132d1af7ef6126884b66b48887a1cf94b293af01e5907dc2a7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/deduplication.py", "bytes": 49745, "sha256": "f8353a8da887fc45823b9174f4578cd816b341d52c3a074964c1c2310af1573e"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/difficulty.py", "bytes": 16760, "sha256": "0b97e060fa6a3b437024a45c42cb131526d5a48b74dd858e4fc0259301f7a6ba"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/display.py", "bytes": 3976, "sha256": "d2dfff4ebe0ff655a240b5c1f073e85702869f618c0aa5c95cc6b54b20be47c8"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/edit.py", "bytes": 2573, "sha256": "d34acdff4530ed211325117395cf7fa9d085f7ee84250d02acf9c8533657a956"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/evaluate.py", "bytes": 25197, "sha256": "96a46f49267ffcf861f7f90b98efbe828d03f27553ceefce745819028589de1e"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/__init__.py", "bytes": 1420, "sha256": "992e9fec50fabd65a003769fc4289dad787b22ca7b6045ce95bf61d3b5e35d0d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/embedding_utils.py", "bytes": 16080, "sha256": "c0970058b7e069bd4493bd86beb68d29047eb0e51d38814a1a146c428e243bf0"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/graph_utils.py", "bytes": 19051, "sha256": "ab2fc0349207baf0c37423f9a28a550923a0ac553c4de2cf0e58447e96ad3050"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/retrieval_utils.py", "bytes": 34709, "sha256": "65fbdddccef339ee74b6434f254e6a94a5d878aab1cc16a432906944e9dcb8e7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/__init__.py", "bytes": 1079, "sha256": "ca5053d64fc35f7d457974879656e63fa7980a7b36d37a28bbe7391f1fa90fe5"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/agent_runner.py", "bytes": 45475, "sha256": "3a25b4da391dda93418a608574bac21bd4c6e12be0d144a2fcb8ae178674c0b7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/container_setup.py", "bytes": 40661, "sha256": "3192acdce35f2d0bd0b191186851b46c251390af16b7f139a1640e16fb1d862c"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/sample_verification.py", "bytes": 20399, "sha256": "0bbd4322534442420a4f782ffb76a5467d6a9d9bd7c32fe212269f304e05fa41"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/verification.py", "bytes": 26984, "sha256": "766e52c34da16cfdc73fc6df22848ab5b9d6b80943d61e7173a844e8efc6c1d9"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/__init__.py", "bytes": 793, "sha256": "34ecd27654b031cd3ede1cead44ed0e8f72bbfe6ffea24c22bfe66d493edbd97"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/discovery.py", "bytes": 5681, "sha256": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/registry.py", "bytes": 7374, "sha256": "69e62c69d461a11f1655285652872005aee11aa549beaec331742dc742593ca3"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/task.py", "bytes": 7693, "sha256": "0b9d705ce759645bd1ee59bb28662a25a951990c024bfeb1e00a41f08350812a"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/results.py", "bytes": 5200, "sha256": "4705de95a05ae5f01f743cbb97046fb0f47f1407f4cd1db249489b2253a7f72f"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/__init__.py", "bytes": 829, "sha256": "ba3725efee8f6d0fba2321851d66d4497ae3cda5863edc0d4a8ef232f1fdb775"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/base.py", "bytes": 10113, "sha256": "aab491aabc84105963bead16dab4d3d5745e31727ba248fd5a568d8f4e0c7dc1"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/docker.py", "bytes": 34531, "sha256": "cc10c287acf7c9326f622e91b98207bb9cdcf4fadf5d4aa5f5658e854051d09d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/subprocess.py", "bytes": 21429, "sha256": "a26e9265d93f0387dd0a820ea451d1c530884bebca41583738f3ab2cc31df1f2"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/submission.py", "bytes": 3577, "sha256": "d387b402223300d06eb6f4f960faf82e997098050f5317961c64e6b95365974a"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/__init__.py", "bytes": 1632, "sha256": "e19d1e81d90976fda93f346104d7f34f74018747ac81838a363de2e357358c31"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/base.py", "bytes": 3768, "sha256": "535a0e6d121549b41906685d0e440e8709c4b8b434bc31dcc82fa7740ec66281"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/execution.py", "bytes": 6573, "sha256": "ff4df981ae1070334665ce8ab726c231e139ac5cf34bb5ebc4660eada28a7b2d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/graph.py", "bytes": 16816, "sha256": "74b74d151925f99ffa37fd68513c234e25b74136369c3a3b072519d36af577cd"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/workspace.py", "bytes": 12316, "sha256": "d38ba2f68636c5e33c9f607bd9e5401dbf81d77125b966ef87c799a1615b05c6"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/warnings.py", "bytes": 1084, "sha256": "f24d5cf69977bd88a5bb82639767ba41c12fed3bd8692969cb7c80051fbb1945"}], "expected_core_wheels": {"adk-eval-core": {"version": "0.1.0", "sha256": "f22cdc3fecb3a340f2aa0dde99024ecc9a82128fe91fb10d3e252564b188b289"}, "adk-submission": {"version": "0.2.13", "sha256": "8a19afc752aa6cec4e98e25b09aa1fe4b020ad902af8f014d294c85d70247d5d"}, "swegemma": {"version": "0.2.10", "sha256": "13261f9d5581840f7600d0d66aea10d6a3e4b53661409e072e55f253436c1252"}}}')
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    locations = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
    if len(locations) != 1: raise ValueError('Attach exactly one frozen matched phone bundle')
    BUNDLE_ROOT = locations[0].parent
    BUNDLE = json.loads(locations[0].read_bytes())
    PHONE_PLAN = json.loads((BUNDLE_ROOT/'phone_plan.json').read_bytes())
    if (BUNDLE['format'] != 'paperbench_cpu_bundle_v1' or len(BUNDLE['cases']) != 4
            or PHONE_PLAN['format'] != 'matched_phone_progress_pilot_v1' or len(PHONE_PLAN['schedule']) != 4
            or hashlib.sha256((BUNDLE_ROOT/'phone_plan.json').read_bytes()).hexdigest() != BUNDLE['plan_sha256']):
        raise ValueError('Lost frozen assignment or phone plan')
    RUN = Path('/kaggle/working')/('matched_phone_grading_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    (RUN/'plan.json').write_text(json.dumps(PLAN,indent=2)+'\n')
    (RUN/'phone_plan.json').write_bytes((BUNDLE_ROOT/'phone_plan.json').read_bytes())
    (RUN/'bundle_manifest.json').write_bytes(locations[0].read_bytes())
    export_run(RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        core_wheels = {r['distribution']: {'version':r['version'],'sha256':r['sha256']}
            for r in selected if r['distribution'] in plan['expected_core_wheels']}
        if core_wheels != plan['expected_core_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise ValueError('Organizer wheels changed again; preserve this attempt before install')
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    prepare_declared_environment(WHEELHOUSE_DIR,RUN,PLAN,export_run,'cpu_controls')
    for filename,key in [('tasks.jsonl','tasks_sha256'),('HARNESS_README.md','guide_sha256')]:
        if hashlib.sha256((DATA_DIR/filename).read_bytes()).hexdigest() != PLAN[key]:
            raise ValueError('Official grading data changed')


In [ ]:
"""Install genuine test dependencies offline without replacing the verifier."""
import hashlib
import importlib
import importlib.metadata as metadata
import io
import json
from pathlib import Path
import platform
import subprocess
import sys
import zipfile


def verify_dependency_bundle(payload: bytes, manifest: dict) -> dict[str, bytes]:
    if hashlib.sha256(payload).hexdigest() != manifest['bundle_sha256']:
        raise ValueError('Dependency bundle hash mismatch')
    expected = {row['file']: row for row in manifest['files']}
    if len(expected) != len(manifest['files']):
        raise ValueError('Duplicate dependency manifest member')
    if any(Path(name).name != name or '/' in name or '\\' in name or not name.endswith('.whl')
           for name in expected):
        raise ValueError('Dependency wheels must be plain basenames')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dependency archive/manifest member mismatch')
        if sum(item.file_size for item in archive.infolist()) > 16 * 1024 * 1024:
            raise ValueError('Unexpected dependency archive size')
        result = {}
        for item in archive.infolist():
            if item.is_dir() or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Invalid dependency archive member')
            data = archive.read(item.filename)
            row = expected[item.filename]
            if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                raise ValueError('Dependency wheel identity mismatch: ' + item.filename)
            result[item.filename] = data
        return result


def install_offline_test_dependencies(payload: bytes, manifest: dict, run_root: Path, checkpoint_exporter) -> dict:
    if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
        raise ValueError('This dependency bundle targets the Kaggle Linux x86_64 runtime')
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise ValueError('This bundle contains wheels for Python 3.12 and 3.13 only')
    wheels = verify_dependency_bundle(payload, manifest)
    profiles = {
        'httpbin_current': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4'],
        'httpbin_legacy_redirect': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4',
                                   'Flask==2.2.5', 'Werkzeug==2.2.3'],
    }
    profile = manifest.get('profile', 'httpbin_current')
    requirements = profiles.get(profile)
    if requirements is None or manifest['requirements'] != requirements:
        raise ValueError('Unexpected test-environment requirements')
    stage = Path('/tmp') / (run_root.name + '_dependency_wheels')
    stage.mkdir(exist_ok=False)
    for name, data in wheels.items():
        (stage / name).write_bytes(data)
    packages = sorted({row['package'] for row in manifest['files']} |
                      {'swegemma', 'adk-submission', 'adk-eval-core', 'vllm', 'torch', 'pytest'})
    def versions():
        result = {}
        for name in packages:
            try:
                result[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                result[name] = None
        return result
    before = versions()
    report = {'scope':'supplemental_offline_test_environment_not_agent_or_private_scoring',
              'python_version':sys.version, 'platform':platform.platform(),
              'bundle_sha256':manifest['bundle_sha256'], 'requirements':requirements, 'profile':profile,
              'before':before, 'model_inference_executed':False,
              'organizer_verifier_replaced':False, 'test_assertions_changed':False}
    (run_root / 'dependency_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
    process = subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        '--no-index','--find-links',str(stage),*requirements],capture_output=True,text=True)
    (run_root / 'dependency_install_stdout.txt').write_text(process.stdout,encoding='utf-8')
    (run_root / 'dependency_install_stderr.txt').write_text(process.stderr,encoding='utf-8')
    importlib.invalidate_caches()
    after = versions()
    report.update(return_code=process.returncode,after=after,
        changed_packages={name:{'before':before[name],'after':after[name]}
                          for name in packages if before[name]!=after[name]},
        pytest_entry_points=[{'name':entry.name,'value':entry.value,'distribution':entry.dist.name}
                             for entry in metadata.entry_points(group='pytest11')])
    core_names = ('swegemma','adk-submission','adk-eval-core','vllm','torch','pytest')
    report['organizer_core_versions_unchanged'] = all(before[name]==after[name] for name in core_names)
    report['httpbin_plugin_registered'] = any(entry['value']=='pytest_httpbin.plugin'
                                            for entry in report['pytest_entry_points'])
    report['status'] = ('installed' if process.returncode==0 and report['organizer_core_versions_unchanged']
                        and report['httpbin_plugin_registered'] else 'installation_not_verified')
    (run_root/'dependency_install_report.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
    checkpoint_exporter(run_root)
    if report['status'] != 'installed':
        raise RuntimeError('Offline dependency installation not verified; preserve its real logs')
    print('OFFLINE_TEST_DEPENDENCIES ' + json.dumps(report),flush=True)
    return report

"""Install a frozen genuine snapshot plugin without changing the organizer core."""
import base64
import hashlib
import importlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys
import tempfile


def install_snapshot_dependencies(manifest, encoded_wheels, run_root, checkpoint_exporter):
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    if sys.platform != 'linux' or sys.version_info[:2] != (3, 12):
        raise ValueError('This separately frozen environment requires Linux CPython3.12')
    if {r['file'] for r in manifest['files']} != set(encoded_wheels):
        raise ValueError('Snapshot dependency membership differs')
    core = ('swegemma', 'adk-submission', 'adk-eval-core', 'vllm', 'torch', 'pytest', 'rich')
    names = sorted(set(core) | {r['package'] for r in manifest['files']})
    def versions():
        result = {}
        for name in names:
            try:
                result[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                result[name] = None
        return result
    before = versions()
    report = {'scope': 'supplemental_snapshot_dependencies_not_agent_or_private_scoring',
              'before': before, 'requirements': manifest['requirements'],
              'organizer_verifier_replaced': False, 'test_assertions_changed': False,
              'model_inference_executed': False}
    with tempfile.TemporaryDirectory(prefix='neural_swe_snapshot_wheels_') as temporary:
        folder = Path(temporary)
        paths = []
        for row in manifest['files']:
            name = row['file']
            if Path(name).name != name or '/' in name or '\\' in name or not name.endswith('.whl'):
                raise ValueError('Snapshot wheel must be a plain filename')
            data = base64.b64decode(encoded_wheels[name], validate=True)
            if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                raise ValueError('Snapshot wheel differs from its frozen PyPI identity')
            path = folder / name
            path.write_bytes(data)
            paths.append(str(path))
        process = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
            '--no-deps', '--force-reinstall', *paths], capture_output=True, text=True, timeout=180)
    (run_root / 'snapshot_dependency_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n', encoding='utf-8')
    (run_root / 'snapshot_install_stdout.txt').write_text(process.stdout or '', encoding='utf-8')
    (run_root / 'snapshot_install_stderr.txt').write_text(process.stderr or '', encoding='utf-8')
    importlib.invalidate_caches()
    after = versions()
    unsatisfied = []
    for row in manifest['files']:
        if after[row['package']] != row['version']:
            unsatisfied.append(row['package'] + ' version differs')
        for text in row['requires_dist']:
            requirement = Requirement(text)
            if requirement.marker is not None and not requirement.marker.evaluate({'extra': ''}):
                continue
            try:
                installed = metadata.version(canonicalize_name(requirement.name))
            except metadata.PackageNotFoundError:
                installed = None
            if installed is None or installed not in requirement.specifier:
                unsatisfied.append(text)
    unchanged = all(before[name] == after[name] for name in core)
    report.update(return_code=process.returncode, after=after,
        changed_packages={name: {'before': before[name], 'after': after[name]}
                          for name in names if before[name] != after[name]},
        organizer_core_versions_unchanged=unchanged, unsatisfied_requirements=unsatisfied,
        status='installed' if process.returncode == 0 and unchanged and not unsatisfied else 'installation_not_verified')
    (run_root / 'snapshot_dependency_install_report.json').write_text(json.dumps(report, indent=2)+'\n', encoding='utf-8')
    checkpoint_exporter(run_root)
    if report['status'] != 'installed':
        raise RuntimeError('Snapshot dependency installation is not verified; preserve receipts')
    return report


In [ ]:
DEPENDENCY_MANIFEST = json.loads('{"scope": "supplemental_test_dependencies_not_agent_submission", "created_utc": "2026-10-03T19:45:18.893670+00:00", "profile": "httpbin_legacy_redirect", "requirements": ["pytest-httpbin==2.1.0", "httpbin==0.10.4", "Flask==2.2.5", "Werkzeug==2.2.3"], "target": "Linux x86_64 CPython 3.12/3.13", "bundle_sha256": "fe32aa50e4fb0f1c4886e2c320382744d6022d07296035e1b72259f7698769f2", "files": [{"file": "blinker-1.9.0-py3-none-any.whl", "package": "blinker", "version": "1.9.0", "bytes": 8458, "sha256": "ba0efaa9080b619ff2f3459d1d500c57bddea4a6b424b60a91141db6fd2f08bc", "source_url": "https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/blinker/1.9.0/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "brotlicffi", "version": "1.2.0.2", "bytes": 1536851, "sha256": "22916101de0e7ff535f2edf54b52a85591853b8ae9a98737643defdd3c063a3a", "source_url": "https://files.pythonhosted.org/packages/06/78/076419ed6c2c6aa3eaac6fd6b076502b4be89d50625fcdc513cd4aeca718/brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/brotlicffi/1.2.0.2/json", "requires_python": ">=3.8", "requires_dist": ["cffi>=1.0.0; python_version < \\"3.13\\"", "cffi>=1.17.0; python_version >= \\"3.13\\""]}, {"file": "cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221822, "sha256": "c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf", "source_url": "https://files.pythonhosted.org/packages/b1/db/dceb9dd5b231e1da801793f8acc9f3c52a7e1afe40bb1aae37e02b0faad5/cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221824, "sha256": "a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2", "source_url": "https://files.pythonhosted.org/packages/95/95/86342356ff5953b3fb06f7ef7c5bee212d45e770abc7218d451b9148313c/cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "click-8.5.0-py3-none-any.whl", "package": "click", "version": "8.5.0", "bytes": 125251, "sha256": "255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360", "source_url": "https://files.pythonhosted.org/packages/58/50/6c0d534c5f134586a8e1ba4e330569e32f057e33372ae556463212fb4cd3/click-8.5.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/click/8.5.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "decorator-5.3.1-py3-none-any.whl", "package": "decorator", "version": "5.3.1", "bytes": 10365, "sha256": "f47fe6fdbd2edd623ecfe36875d37aba411624e2670dd395dddae1358689bb3c", "source_url": "https://files.pythonhosted.org/packages/05/7f/798705f5296a58ca505d600456748d1be48078eac8a7050d8a98bc9edb89/decorator-5.3.1-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/decorator/5.3.1/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "Flask-2.2.5-py3-none-any.whl", "package": "Flask", "version": "2.2.5", "bytes": 101817, "sha256": "58107ed83443e86067e41eff4631b058178191a355886f8e479e347fa1285fdf", "source_url": "https://files.pythonhosted.org/packages/9f/1a/8b6d48162861009d1e017a9740431c78d860809773b66cac220a11aa3310/Flask-2.2.5-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Flask/2.2.5/json", "requires_python": ">=3.7", "requires_dist": ["Werkzeug (>=2.2.2)", "Jinja2 (>=3.0)", "itsdangerous (>=2.0)", "click (>=8.0)", "importlib-metadata (>=3.6.0) ; python_version < \\"3.10\\"", "asgiref (>=3.2) ; extra == \'async\'", "python-dotenv ; extra == \'dotenv\'"]}, {"file": "greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 621404, "sha256": "975736b002ed080d124cf81a79cb7e05cb26d6b3f5c7a7b651c0fcce70353aa1", "source_url": "https://files.pythonhosted.org/packages/66/6a/1594f3869c57c149abdb380492529e04d4c0229b5e4d79572c5bd0aaa673/greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 623779, "sha256": "f98e8215e172f567ce80eeaed9107fb4d32b6c44f26983d9b8334658136a205a", "source_url": "https://files.pythonhosted.org/packages/85/cb/ab0c123c514ed4e94c0dc9ee2e86362633e6b998cfc05de7fc9ac2eb9690/greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "httpbin-0.10.4-py2.py3-none-any.whl", "package": "httpbin", "version": "0.10.4", "bytes": 100765, "sha256": "71862093f371c3a39cfcdcd2f073aaf610b6433f9d4c0130248e676453fca65d", "source_url": "https://files.pythonhosted.org/packages/e5/39/e9fc7334c69f82c307ae861e13704437d31b71514d273061e08db5830b92/httpbin-0.10.4-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/httpbin/0.10.4/json", "requires_python": ">=3.8", "requires_dist": ["brotlicffi", "decorator", "flask>=2.2.4", "greenlet<3.0; python_version < \\"3.12\\"", "greenlet>=3.0.0a1; python_version >= \\"3.12.0rc0\\"", "importlib-metadata; python_version < \\"3.8\\"", "six", "werkzeug>=2.2.2", "pytest; extra == \\"test\\"", "tox; extra == \\"test\\"", "flasgger; extra == \\"mainapp\\"", "gunicorn; extra == \\"mainapp\\"", "gevent; extra == \\"mainapp\\""]}, {"file": "itsdangerous-2.2.0-py3-none-any.whl", "package": "itsdangerous", "version": "2.2.0", "bytes": 16234, "sha256": "c6242fc49e35958c8b15141343aa660db5fc54d4f13a1db01a3f5891b98700ef", "source_url": "https://files.pythonhosted.org/packages/04/96/92447566d16df59b2a776c0fb82dbc4d9e07cd95062562af01e408583fc4/itsdangerous-2.2.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/itsdangerous/2.2.0/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "jinja2-3.1.6-py3-none-any.whl", "package": "Jinja2", "version": "3.1.6", "bytes": 134899, "sha256": "85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67", "source_url": "https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Jinja2/3.1.6/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe>=2.0", "Babel>=2.7 ; extra == \\"i18n\\""]}, {"file": "markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 22985, "sha256": "8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581", "source_url": "https://files.pythonhosted.org/packages/4f/a7/aeedb5140afa41fc74c225e9184ab96723a6e873b6ee1c9fede7283456d8/markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 23017, "sha256": "434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6", "source_url": "https://files.pythonhosted.org/packages/63/e0/cec6865dfe88cb48fedd4b20aed6af5158e41092adcbf3e028bcc6ec2108/markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "pycparser-3.0-py3-none-any.whl", "package": "pycparser", "version": "3.0", "bytes": 48172, "sha256": "b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992", "source_url": "https://files.pythonhosted.org/packages/0c/c3/44f3fbbfa403ea2a7c779186dc20772604442dde72947e7d01069cbe98e3/pycparser-3.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pycparser/3.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "pytest_httpbin-2.1.0-py3-none-any.whl", "package": "pytest-httpbin", "version": "2.1.0", "bytes": 8309, "sha256": "b3bf7346cc2ad231447189cd85b458e341056684a7a69d69533dd29692209cdd", "source_url": "https://files.pythonhosted.org/packages/0d/97/149289c7123f75e98edc94872545c2f597f349ee037b1e0a47aef0362fd4/pytest_httpbin-2.1.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pytest-httpbin/2.1.0/json", "requires_python": ">=3.8", "requires_dist": ["httpbin", "requests; extra == \\"test\\"", "pytest; extra == \\"test\\""]}, {"file": "six-1.17.0-py2.py3-none-any.whl", "package": "six", "version": "1.17.0", "bytes": 11050, "sha256": "4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274", "source_url": "https://files.pythonhosted.org/packages/b7/ce/149a00dd41f10bc29e5921b496af8b574d8413afcd5e30dfa0ed46c2cc5e/six-1.17.0-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/six/1.17.0/json", "requires_python": ">=2.7, !=3.0.*, !=3.1.*, !=3.2.*", "requires_dist": []}, {"file": "Werkzeug-2.2.3-py3-none-any.whl", "package": "Werkzeug", "version": "2.2.3", "bytes": 233551, "sha256": "56433961bc1f12533306c624f3be5e744389ac61d722175d543e1751285da612", "source_url": "https://files.pythonhosted.org/packages/f6/f8/9da63c1617ae2a1dec2fbf6412f3a0cfe9d4ce029eccbda6e1e4258ca45f/Werkzeug-2.2.3-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Werkzeug/2.2.3/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe (>=2.1.1)", "watchdog ; extra == \'watchdog\'"]}], "verifier_or_test_assertion_changes": false, "dependency_rationale": "Werkzeug 2.2.3 preserves ASCII Location headers via safe_conversion; Flask 2.2.5 supports that version. The Python 3.12-compatible httpbin plugin remains unchanged. This is a supplemental compatibility profile, not the historical requirements lock or proof of equivalence to private scoring.", "source_references": ["https://raw.githubusercontent.com/psf/requests/0b4d494192de489701d3a2e32acef8fb5d3f042e/requirements-dev.txt", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/wrappers/response.py", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/urls.py"]}')
DEPENDENCY_INPUT_PREFIX = 'httpbin_legacy_redirect'
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    # Attach the private NeuralSWE Offline Test Dependencies dataset. Package
    # binaries are input files, not notebook source (Kaggle limits source to 1 MB).
    matches = list(Path('/kaggle/input').rglob(DEPENDENCY_INPUT_PREFIX+'_manifest.json'))
    if len(matches) != 1:
        raise RuntimeError('Attach exactly one version of the supplemental dependency dataset')
    input_root = matches[0].parent
    if json.loads(matches[0].read_text()) != DEPENDENCY_MANIFEST:
        raise ValueError('Attached dependency manifest differs from the frozen source')
    archive_path = input_root/(DEPENDENCY_INPUT_PREFIX+'_wheels.zip')
    if archive_path.is_file():
        dependency_payload = archive_path.read_bytes()
    else:
        # Kaggle may expand uploaded ZIPs. Recreate the same deterministic
        # transport archive from byte-verified original wheels without alteration.
        stream = io.BytesIO()
        with zipfile.ZipFile(stream,'w',compression=zipfile.ZIP_STORED) as archive:
            for row in DEPENDENCY_MANIFEST['files']:
                locations = list(input_root.rglob(row['file']))
                locations = [p for p in locations if
                    DEPENDENCY_INPUT_PREFIX+'_wheels' in p.parts]
                if len(locations) != 1:
                    raise ValueError('Missing or ambiguous genuine dependency wheel: '+row['file'])
                data = locations[0].read_bytes()
                if len(data)!=row['bytes'] or hashlib.sha256(data).hexdigest()!=row['sha256']:
                    raise ValueError('Attached dependency wheel changed')
                item=zipfile.ZipInfo(row['file'],date_time=(1980,1,1,0,0,0))
                item.create_system=0  # Match the recorded Windows transport metadata.
                item.external_attr=0o100644 << 16
                archive.writestr(item,data)
        dependency_payload = stream.getvalue()
    dependency_report = install_offline_test_dependencies(
        dependency_payload,DEPENDENCY_MANIFEST,RUN,export_run)
else:
    print('Local definitions only; no test-environment installation performed')

SNAPSHOT_MANIFEST = json.loads('{"scope": "genuine_supplemental_test_dependencies_not_agent_or_private_scoring", "files": [{"file": "asttokens-3.0.1-py3-none-any.whl", "package": "asttokens", "version": "3.0.1", "bytes": 27047, "sha256": "15a3ebc0f43c2d0a50eeafea25e19046c68398e487b9f1f5b517f7c0f40f976a", "source_url": "https://files.pythonhosted.org/packages/d2/39/e7eaf1799466a4aef85b6a4fe7bd175ad2b1c6345066aa33f1f58d4b18d0/asttokens-3.0.1-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/asttokens/3.0.1/json", "requires_dist": ["astroid<5,>=2; extra == \\"astroid\\"", "astroid<5,>=2; extra == \\"test\\"", "pytest<9.0; extra == \\"test\\"", "pytest-cov; extra == \\"test\\"", "pytest-xdist; extra == \\"test\\""]}, {"file": "executing-2.2.1-py2.py3-none-any.whl", "package": "executing", "version": "2.2.1", "bytes": 28317, "sha256": "760643d3452b4d777d295bb167ccc74c64a81df23fb5e08eff250c425a4b2017", "source_url": "https://files.pythonhosted.org/packages/c1/ea/53f2148663b321f21b5a606bd5f191517cf40b7072c0497d3c92c4a13b1e/executing-2.2.1-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/executing/2.2.1/json", "requires_dist": ["asttokens>=2.1.0; extra == \\"tests\\"", "ipython; extra == \\"tests\\"", "pytest; extra == \\"tests\\"", "coverage; extra == \\"tests\\"", "coverage-enable-subprocess; extra == \\"tests\\"", "littleutils; extra == \\"tests\\"", "rich; python_version >= \\"3.11\\" and extra == \\"tests\\""]}, {"file": "inline_snapshot-0.36.0-py3-none-any.whl", "package": "inline-snapshot", "version": "0.36.0", "bytes": 98033, "sha256": "1079cf302098fedb418c7ac2be9cadd14f2e71461143bcaf605be1214530bef4", "source_url": "https://files.pythonhosted.org/packages/39/de/f625b9c923c4264ee8d94e416c0d5fd59194f7b6cd8fc0d6ec2f51dfc506/inline_snapshot-0.36.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/inline-snapshot/0.36.0/json", "requires_dist": ["asttokens>=2.0.5", "executing>=2.2.0", "pytest>=8.3.4", "rich>=13.7.1", "tomli>=2.0.0; python_version < \'3.11\'", "typing-extensions", "black>=23.3.0; extra == \'black\'", "dirty-equals>=0.9.0; extra == \'dirty-equals\'"]}], "requirements": ["inline-snapshot==0.36.0", "asttokens==3.0.1", "executing==2.2.1"], "assertions_changed": false}')
SNAPSHOT_WHEELS = {'asttokens-3.0.1-py3-none-any.whl': '...', 'executing-2.2.1-py2.py3-none-any.whl': '...', 'inline_snapshot-0.36.0-py3-none-any.whl': '...'}
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    install_snapshot_dependencies(SNAPSHOT_MANIFEST, SNAPSHOT_WHEELS, RUN, export_run)


In [ ]:
"""Current-contract independent grading of the frozen matched phone pilot.

The older mobile_patch_validation.py and original grade_request.json remain
unchanged. This separately named request is reconstructed from the same actual
native bytes against the organizer versions qualified in CPU79, not an upgrade
of old grades. It performs no model inference or phone action on import.
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path, PurePosixPath
import time

EXPECTED_CORE_PACKAGES = {"swegemma": "0.2.10", "adk-submission": "0.2.13", "adk-eval-core": "0.1.0"}
TASKS_SHA256 = "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6"
GUIDE_SHA256 = "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb"
VERIFY_TASK_SHA256 = "dc89c3595217c5627d73dcccf41b90a45a2743f2f770537e2f5d2da0b3bced16"


def sha_file(path):
    value = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1048576):
            value.update(chunk)
    return value.hexdigest()


def installed_versions(names):
    values = {}
    for name in names:
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def bound_file(root, name):
    relative = PurePosixPath(name)
    if (not name or "\\" in name or ":" in name or relative.is_absolute()
            or any(part in {"", ".", ".."} for part in name.split("/"))):
        raise ValueError("Invalid evidence member path")
    result = root.joinpath(*relative.parts)
    if result.is_symlink() or not result.resolve().is_relative_to(root.resolve()):
        raise ValueError("Evidence member escaped its root")
    return result


def validate_request(root, request):
    if request["format"] != "native_mobile_grade_request_v1" or request["scope"] != "public_development_not_held_out":
        raise ValueError("Unexpected grading request scope")
    if (request["expected_core_packages"] != EXPECTED_CORE_PACKAGES
            or request["test_timeout_seconds"] != 300 or request["tasks_sha256"] != TASKS_SHA256
            or request["guide_sha256"] != GUIDE_SHA256 or request["expected_verify_task_sha256"] != VERIFY_TASK_SHA256):
        raise ValueError("Grading request differs from the observed organizer contract")
    for name, expected in request["source_sha256"].items():
        if sha_file(bound_file(root, name)) != expected:
            raise ValueError("Mobile evidence changed: " + name)
    def read(name):
        if name not in request["source_sha256"]:
            raise ValueError("Unbound mobile evidence member")
        return json.loads(bound_file(root, name).read_text(encoding="utf-8"))
    assignment = read("assignment.json")
    summary = read("summary.json")
    episode = read("episode.json")
    reconstruction = read(request["reconstruction_path"])
    arm = assignment.get("progress_policy")
    if (arm not in ("enabled", "adaptive_baseline") or summary.get("progress_policy") != arm
            or episode.get("progress_policy") != arm):
        raise ValueError("Native matched-policy arm differs or is missing")
    task_id = request["task_id"]
    if (assignment["task_id"] != task_id or summary["task_id"] != task_id or episode["task_id"] != task_id
            or assignment["run_id"] != request["run_id"] or episode["run_id"] != request["run_id"]
            or summary["run_id"] != request["run_id"]
            or summary["run_identity_matches"] is not True
            or reconstruction["task_id"] != task_id
            or reconstruction["repo"] != request["repo"] or reconstruction["base_commit"] != request["base_commit"]
            or reconstruction["input_manifest_sha256"] != assignment["input_manifest_sha256"]
            or assignment["input_manifest"] != episode["input_manifest"]
            or (assignment["input_manifest"]["task_id"], assignment["input_manifest"]["repo"], assignment["input_manifest"]["base_commit"]) !=
               (task_id, request["repo"], request["base_commit"])
            or reconstruction["reconstructed_tree_matches_device_inventory"] is not True):
        raise ValueError("Native assignment/export identities differ")
    patch_path = request["patch_path"]
    if patch_path not in request["source_sha256"]:
        raise ValueError("Unbound generated patch")
    patch = bound_file(root, patch_path).read_bytes()
    if hashlib.sha256(patch).hexdigest() != reconstruction["patch_sha256"] or len(patch) != reconstruction["patch_bytes"]:
        raise ValueError("Generated patch differs from reconstructed device bytes")
    if reconstruction.get("export_origin") == "host_recovery_of_quiesced_native_workspace_not_native_export_completion":
        recovery = read("recovery/recovery_manifest.json")
        required = {"workspace_after_stop.tar", "recovery/recovered_changes.zip"}
        if not required.issubset(request["source_sha256"]):
            raise ValueError("Recovered working bytes are not bound to grading")
        if (assignment.get("workspace_recovery_quiesced") is not True
                or recovery["process_quiesced_by_runner"] is not True
                or recovery["run_id"] != request["run_id"] or recovery["task_id"] != task_id
                or recovery["native_episode_sha256"] != request["source_sha256"]["episode.json"]
                or recovery["input_manifest_sha256"] != assignment["input_manifest_sha256"]
                or recovery["workspace_tar_sha256"] != request["source_sha256"]["workspace_after_stop.tar"]
                or recovery["recovered_export_sha256"] != request["source_sha256"]["recovery/recovered_changes.zip"]
                or reconstruction["device_export_sha256"] != recovery["recovered_export_sha256"]
                or recovery["generation_completion_established_by_recovery"] is not False):
            raise ValueError("Recovered workspace provenance differs from its native assignment")
    return patch.decode("utf-8"), assignment, summary


def classify_patch(clean, candidate, reference, patch_bytes):
    def passed(row):
        return row.get("resolved") is True and row.get("test_exit_code") == 0
    known_fixture = passed(reference) and clean.get("resolved") is False and clean.get("test_exit_code") in (1, 2)
    return {"generated_patch_passes_official_tests": passed(candidate),
            "clean_base_passes": passed(clean), "reference_passes": passed(reference),
            "fixture_discriminates": known_fixture,
            "discriminating_patch_success": bool(known_fixture and passed(candidate) and patch_bytes > 0),
            "model_inference_established_by_grading": False,
            "full_workflow_deadline_verified": None, "official_leaderboard_score": None}


def run_mobile_patch_validation(data_dir, episode_root, run_root, request, checkpoint_exporter, environment_receipt=None):
    patch, assignment, native_summary = validate_request(episode_root, request)
    if sha_file(data_dir / "tasks.jsonl") != request["tasks_sha256"] or sha_file(data_dir / "HARNESS_README.md") != request["guide_sha256"]:
        raise ValueError("Organizer data differs from frozen grading request")
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    if torch.cuda.device_count() != 0:
        raise ValueError("Select Accelerator None: this is CPU patch validation, not model inference")
    packages = {name: importlib.metadata.version(name) for name in EXPECTED_CORE_PACKAGES}
    if packages != request["expected_core_packages"]:
        raise ValueError("Organizer core package versions differ from frozen request")
    task = next(item for item in load_tasks(data_dir / "tasks.jsonl") if item.instance_id == request["task_id"])
    if task.repo != request["repo"] or task.base_commit != request["base_commit"]:
        raise ValueError("Organizer task differs from native assignment")
    # Read the reference only in the evaluator after generation is frozen. Never
    # print or export its text, and never count it as a generated solution.
    reference_row = next(row for row in map(json.loads, (data_dir / "tasks.jsonl").read_text(encoding="utf-8").splitlines())
                         if row["instance_id"] == request["task_id"])
    reference_patch = reference_row["patch"]
    module = inspect.getmodule(Evaluator)
    run_root.mkdir(parents=True, exist_ok=False)
    verifier_source = inspect.getsource(module.verify_task)
    (run_root / "verify_task_api.py").write_text(verifier_source, encoding="utf-8")
    if hashlib.sha256(verifier_source.encode()).hexdigest() != request["expected_verify_task_sha256"]:
        raise ValueError("Official verifier source differs from the recorded real CPU fixture run")
    snapshot, base_snapshot, task_patch = module.resolve_task_snapshot_paths(data_dir / "snapshots", task.instance_id, task.repo)
    snapshots = {str(path.relative_to(data_dir)): sha_file(path) for path in (snapshot, base_snapshot, task_patch) if path is not None}
    report = {"scope": "official_public_cpu_validation_of_frozen_mobile_patch_not_leaderboard",
              "created_utc": datetime.now(timezone.utc).isoformat(), "run_id": request["run_id"],
              "task_id": task.instance_id, "assigned_native_tasks": 1, "assigned_grading_conditions": 3,
              "grading_request": request, "status": "started", "observations": [],
              "generator_reported_status": native_summary,
              "model_inference_executed_by_grader": False, "gpu_count": 0,
              "core_packages": packages, "official_snapshot_sha256": snapshots,
              "generator_input_origin": "GitHub_codeload_exact_declared_commit",
              "device_input_tree_equivalence_with_official_snapshot_established": False,
              "private_scoring_environment_equivalence_established": False,
              "supplemental_environment_receipt": environment_receipt,
              "observed_test_packages": installed_versions(("pytest", "Flask", "Werkzeug", "requests", "httpbin", "pytest-httpbin")),
              "verify_task_source_sha256": hashlib.sha256(verifier_source.encode()).hexdigest(),
              "task_correctness": None, "full_workflow_deadline_verified": None,
              "reference_text_exported": False, "official_leaderboard_score": None}
    def save():
        (run_root / "validation.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    save(); checkpoint_exporter(run_root)

    async def evaluate():
        outcomes = {}
        for condition, submitted_patch in (("clean_base", ""), ("native_unchanged_patch", patch), ("official_reference", reference_patch)):
            destination = run_root / condition
            destination.mkdir()
            config = EvalConfig(tasks_path=data_dir / "tasks.jsonl", snapshots_dir=data_dir / "snapshots",
                results_dir=destination, submission_dir=run_root / "unused_agent_directory", models={},
                sandbox="subprocess", timeout_seconds=request["test_timeout_seconds"], wheels_dir=data_dir / "wheels",
                graph_dir=str(data_dir / "graphs"), embeddings_dir=str(data_dir / "embeddings"))
            evaluator = Evaluator(config)
            print("MOBILE_PATCH_VALIDATION_START " + condition, flush=True)
            result = await module.verify_task(evaluator.docker, config, task, snapshot,
                base_snapshot_path=base_snapshot, patch_path=task_patch,
                agent_patch=submitted_patch, start_time=time.perf_counter())
            details = result.model_dump(mode="json", exclude={"trace", "agent_patch"})
            details.update(condition=condition, replayed_patch_sha256=hashlib.sha256(submitted_patch.encode()).hexdigest())
            (destination / "official_task_result.json").write_text(json.dumps(details, indent=2) + "\n", encoding="utf-8")
            (destination / "test_output.txt").write_text(result.test_output or "", encoding="utf-8")
            observed = {"condition": condition, "resolved": result.resolved, "test_exit_code": result.test_exit_code,
                        "duration_seconds": result.duration_seconds, "result_path": condition + "/official_task_result.json"}
            outcomes[condition] = observed
            report["observations"].append(observed)
            save(); print("MOBILE_PATCH_VALIDATION_RESULT " + json.dumps(observed), flush=True)
            checkpoint_exporter(run_root)
        report["classification"] = classify_patch(outcomes["clean_base"], outcomes["native_unchanged_patch"],
                                                    outcomes["official_reference"], len(patch.encode()))
        report["status"] = "completed"
        report["task_correctness"] = report["classification"]["generated_patch_passes_official_tests"]
        save(); checkpoint_exporter(run_root)
        return report
    try:
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(evaluate())).result()
    except Exception as error:
        report.update(status="grading_exception", error_type=type(error).__name__, error=str(error))
        save(); checkpoint_exporter(run_root)
        raise


def prepare_grade_request(episode_root):
    """Freeze a request only from a recovered, hash-bound real episode receipt."""
    assignment = json.loads((episode_root / "assignment.json").read_text(encoding="utf-8"))
    summary = json.loads((episode_root / "summary.json").read_text(encoding="utf-8"))
    if summary["run_identity_matches"] is not True or summary["instrumentation_started"] is not True:
        raise ValueError("No recovered matching native episode exists")
    reconstruction_dirs = [name for name in ("export_reconstruction", "partial_export_reconstruction", "recovery/reconstruction")
                           if (episode_root / name / "reconstruction.json").is_file()]
    if len(reconstruction_dirs) != 1:
        raise ValueError("Require one unchanged reconstructed device export")
    directory = reconstruction_dirs[0]
    reconstruction = json.loads((episode_root / directory / "reconstruction.json").read_text(encoding="utf-8"))
    names = ["assignment.json", "summary.json", "episode.json", directory + "/reconstruction.json", directory + "/generated.patch"]
    names += [name for name in ("memory_samples.jsonl", "model_responses.jsonl", "tool_results.jsonl", "stage_delta.jsonl", "events.jsonl")
              if (episode_root / name).is_file()]
    if directory == "recovery/reconstruction":
        names += ["recovery/recovery_manifest.json", "recovery/recovered_changes.zip", "workspace_after_stop.tar"]
    request = {"format": "native_mobile_grade_request_v1", "scope": "public_development_not_held_out",
               "run_id": assignment["run_id"], "task_id": assignment["task_id"],
               "repo": reconstruction["repo"], "base_commit": reconstruction["base_commit"],
               "source_sha256": {name: sha_file(episode_root / name) for name in names},
               "patch_path": directory + "/generated.patch", "reconstruction_path": directory + "/reconstruction.json",
               "tasks_sha256": TASKS_SHA256, "guide_sha256": GUIDE_SHA256,
               "expected_core_packages": EXPECTED_CORE_PACKAGES, "test_timeout_seconds": 300,
               "expected_verify_task_sha256": VERIFY_TASK_SHA256}
    validate_request(episode_root, request)
    target = episode_root / "grade_request_current.json"
    with target.open("x", encoding="utf-8") as stream:
        stream.write(json.dumps(request, indent=2) + "\n")
    return request


In [ ]:
"""Conservative analysis of frozen phone assignments and independent CPU grades.

Generation completion, a test-process exit, and a verified repair are distinct.
Definitions perform no phone access, inference, dependency installation or grading.
"""
import hashlib
import json
from pathlib import Path
import statistics


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def grade_state(report, patch_bytes, production_paths):
    observations = report.get('observations', [])
    rows = {r['condition']: r for r in observations}
    if len(rows) != len(observations):
        raise ValueError('Duplicate grading condition')
    needed = {'clean_base', 'native_unchanged_patch', 'official_reference'}
    if report.get('status') != 'completed' or set(rows) != needed:
        return dict(repair=None, reason='grading_incomplete', candidate_test_pass=None)
    def passed(row):
        return row.get('resolved') is True and row.get('test_exit_code') == 0
    def known(row):
        return (passed(row) or row.get('resolved') is False and row.get('test_exit_code') == 1)
    # Collection errors (2), setup failures (-1), absent fields and contradictory
    # exit/resolved pairs do not establish a failing candidate or working fixture.
    clean, candidate, reference = [rows[k] for k in ('clean_base', 'native_unchanged_patch', 'official_reference')]
    if not all(known(row) for row in (clean, candidate, reference)):
        return dict(repair=None, reason='grader_result_unknown', candidate_test_pass=None)
    if not passed(reference) or passed(clean):
        return dict(repair=None, reason='fixture_not_discriminating', candidate_test_pass=passed(candidate))
    if patch_bytes == 0 or not production_paths:
        return dict(repair=False, reason='no_production_change', candidate_test_pass=passed(candidate))
    return dict(repair=passed(candidate), reason='independent_discriminating_fixture', candidate_test_pass=passed(candidate))


def summarize(plan, measurements):
    schedule = {r['assignment_id']: r for r in plan['schedule']}
    if len(schedule) != len(plan['schedule']):
        raise ValueError('Duplicate planned assignment')
    actual = {}
    for row in measurements:
        key = row['assignment_id']
        if key not in schedule or key in actual or row['task_id'] != schedule[key]['task_id']:
            raise ValueError('Unplanned, duplicate or mismatched measurement')
        actual[key] = row
    result = []
    for key, expected in schedule.items():
        row = actual.get(key)
        if row is None:
            result.append(dict(assignment_id=key, task_id=expected['task_id'], repeat=expected['repeat'],
                               status='not_attempted', repair=None, generation_completed=None,
                               elapsed_ms=None, cleanup_verified=None))
        else:
            result.append(dict(row, repeat=expected['repeat']))
    timings = [r['elapsed_ms'] for r in result if r.get('generation_completed') is True
               and isinstance(r.get('elapsed_ms'), (int, float))]
    by_task = []
    for task in plan['tasks']:
        rows = [r for r in result if r['task_id'] == task['task_id']]
        repairs = [r.get('repair') for r in rows]
        complete = all(isinstance(v, bool) for v in repairs)
        by_task.append(dict(task_id=task['task_id'], assigned=len(rows),
            verified_repairs=sum(v is True for v in repairs), unknown=sum(v is None for v in repairs),
            repair_outcome_repeatable=(len(set(repairs)) == 1) if complete else None))
    return dict(scope=plan['scope'], assigned=len(result), recorded=len(actual),
        generation_completed=sum(r.get('generation_completed') is True for r in result),
        verified_repairs=sum(r.get('repair') is True for r in result),
        verified_failures=sum(r.get('repair') is False for r in result),
        unknown_repairs=sum(r.get('repair') is None for r in result),
        median_completed_elapsed_ms=statistics.median(timings) if timings else None,
        per_assignment=result, per_task=by_task, population_accuracy_or_harness_superiority_established=False,
        phone_episodes_are_not_independent_tasks=True, prize_quality_established=False)


def feedback_comparison(records):
    """Paired policy coverage on real receipts plus labelled transport controls.

    These controls test evidence attribution, not model repair accuracy. An
    incorrect receipt is unknown, not a failed repair. No inferred gold grades.
    """
    rows = []
    for record in records:
        required = ('patch_identity_matches', 'task_identity_matches', 'source_identity_matches',
                    'verifier_identity_matches', 'test_identity_matches')
        qualified = all(record.get(k) is True for k in required)
        exit_code = record.get('test_exit_code')
        bare = True if exit_code == 0 else False if exit_code == 1 else None
        attributed = bare if qualified else None
        rows.append(dict(receipt_id=record['receipt_id'], scope=record['scope'],
            receipt_attribution_valid=qualified, bare_exit_verdict=bare,
            qualified_verdict=attributed, reference_grade=record.get('reference_grade'),
            bare_exit_unsupported_claim=bare is not None and not qualified,
            qualified_unsupported_claim=attributed is not None and not qualified))
    if len({r['receipt_id'] for r in rows}) != len(rows):
        raise ValueError('Duplicate feedback receipt')
    return dict(paired_receipts=rows, assigned=len(rows),
        bare_exit_unsupported_claims=sum(r['bare_exit_unsupported_claim'] for r in rows),
        qualified_unsupported_claims=sum(r['qualified_unsupported_claim'] for r in rows),
        bare_exit_coverage=sum(r['bare_exit_verdict'] is not None for r in rows),
        qualified_coverage=sum(r['qualified_verdict'] is not None for r in rows),
        limitations='Controlled invalid attribution is not a measured natural fault rate or repair gain.')


In [ ]:
"""Conservative paired analysis; two exposed tasks are not four independent tasks."""
def summarize_matched_phone(plan, measurements):
    assigned = {a['assignment_id']: a for a in plan['schedule']}
    if len(assigned) != 4 or len(assigned) != len(plan['schedule']):
        raise ValueError('Unexpected matched assignment count')
    actual = {}
    for row in measurements:
        key = row['assignment_id']
        if (key not in assigned or key in actual or row['task_id'] != assigned[key]['task_id']
                or row['arm'] != assigned[key]['arm']):
            raise ValueError('Unexpected or duplicated matched measurement')
        actual[key] = row
    rows = [dict(actual.get(key, dict(assignment_id=key, task_id=a['task_id'], arm=a['arm'],
                status='not_attempted', repair=None))) for key,a in assigned.items()]
    pairs = []
    for task in plan['tasks']:
        case = {row['arm']: row for row in rows if row['task_id'] == task['task_id']}
        if set(case) != {'enabled','adaptive_baseline'}: raise ValueError('Lost paired arm')
        baseline, enabled = [case[arm].get('repair') for arm in ('adaptive_baseline','enabled')]
        known = isinstance(baseline,bool) and isinstance(enabled,bool)
        pairs.append(dict(task_id=task['task_id'], both_grades_known=known,
            baseline_repair=baseline, enabled_repair=enabled,
            enabled_only_verified_repair=(enabled and not baseline) if known else None,
            baseline_only_verified_repair=(baseline and not enabled) if known else None))
    arms = {}
    for arm in ('adaptive_baseline','enabled'):
        subset = [row for row in rows if row['arm'] == arm]
        arms[arm] = dict(assigned=len(subset), verified_repairs=sum(r.get('repair') is True for r in subset),
            verified_failures=sum(r.get('repair') is False for r in subset),
            unknown=sum(r.get('repair') is None for r in subset),
            no_change_reported_completed=sum(r.get('no_change_reported_completed') is True for r in subset))
    return dict(scope=plan['scope'], assigned=4, exposed_tasks=2, recorded=len(actual),
        arms=arms, pairs=pairs, per_assignment=rows,
        known_pairs=sum(p['both_grades_known'] for p in pairs),
        enabled_only_verified_repairs=sum(p['enabled_only_verified_repair'] is True for p in pairs),
        baseline_only_verified_repairs=sum(p['baseline_only_verified_repair'] is True for p in pairs),
        population_accuracy_or_causal_component_gain_established=False,
        independent_tasks_are_two_not_four=True, repeated_trials_per_case_and_arm=1,
        expansion_gate_requires_separate_diff_uptake_safety_review=True, prize_quality_established=False)


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    assigned = {a['assignment_id']: a for a in PHONE_PLAN['schedule']}
    if {r['assignment_id'] for r in BUNDLE['cases']} != set(assigned): raise ValueError('Schedule differs')
    measurements, feedback = [], []
    for record in BUNDLE['cases']:
        assignment = assigned[record['assignment_id']]
        if record['task_id'] != assignment['task_id']: raise ValueError('Task differs')
        measurement = dict(assignment_id=record['assignment_id'], task_id=record['task_id'],
            arm=assignment['arm'], status=record['status'], repair=None, generation_completed=None, elapsed_ms=None, cleanup_verified=None)
        if record['status'] == 'frozen_export_ready_for_CPU_grading':
            try:
                for name, expected in record['files'].items():
                    if sha_file(bound_file(BUNDLE_ROOT, name)) != expected: raise ValueError('Frozen phone file changed')
                root = bound_file(BUNDLE_ROOT, record['episode_directory'])
                request = json.loads((root/'grade_request_current.json').read_bytes())
                if sha_file(root/'grade_request_current.json') != record['request_sha256']: raise ValueError('Request changed')
                patch, native_assignment, native_summary = validate_request(root, request)
                task = next(t for t in PHONE_PLAN['tasks'] if t['task_id'] == record['task_id'])
                if (request['task_id'],request['repo'],request['base_commit']) != (task['task_id'],task['repo'],task['base_commit']):
                    raise ValueError('Patch task differs')
                if (native_assignment['input_manifest_sha256'] != task['input_manifest_sha256']
                        or native_assignment['progress_policy'] != assignment['arm']
                        or native_assignment['verified_source_apk_build_binding'] != BUNDLE['frozen_build_identity']):
                    raise ValueError('Native source input differs')
                report = run_mobile_patch_validation(DATA_DIR, root, RUN/'episodes'/record['assignment_id'], request,
                    lambda _: export_run(RUN), environment_receipt=dependency_report)
                reconstruction = json.loads((root/request['reconstruction_path']).read_bytes())
                # Reproduction scripts alone do not constitute production repairs.
                prefixes = {'Textualize/rich':'rich/', 'fastapi/fastapi':'fastapi/'}
                paths = reconstruction['changed_files']+reconstruction['removed_files']
                production = [p for p in paths if p.startswith(prefixes[request['repo']])]
                state = grade_state(report, len(patch.encode()), production)
                measurement.update(repair=state['repair'], grading_reason=state['reason'],
                    generation_completed=native_summary['generation_episode_completed'],
                    production_paths=production, patch_bytes=len(patch.encode()),
                    no_change_reported_completed=(not production and native_summary.get('generation_loop_succeeded') is True),
                    elapsed_ms=native_summary.get('episode_elapsed_including_shutdown_ms'),
                    cleanup_verified=native_summary.get('phone_cleanup_verified'), status=report['status'])
                candidate = next(r for r in report['observations'] if r['condition']=='native_unchanged_patch')
                receipt = dict(receipt_id=record['assignment_id'], scope='actual_independent_grade',
                    test_exit_code=candidate['test_exit_code'], reference_grade=state['candidate_test_pass'],
                    patch_identity_matches=True, task_identity_matches=True, source_identity_matches=True,
                    verifier_identity_matches=True, test_identity_matches=True)
                feedback.append(receipt)
                # Transport negative controls have explicit labels and no new test/model outcomes.
                for field in ('patch_identity_matches','task_identity_matches','source_identity_matches'):
                    feedback.append(dict(receipt, receipt_id=record['assignment_id']+'_'+field,
                        scope='synthetic_attribution_mismatch_control', **{field:False}))
            except Exception as error:
                measurement.update(status='grading_exception_unknown', error_type=type(error).__name__)
        measurements.append(measurement)
        (RUN/'measurements.json').write_text(json.dumps(measurements, indent=2)+'\n')
        analysis = summarize_matched_phone(PHONE_PLAN, measurements)
        (RUN/'analysis.json').write_text(json.dumps(analysis, indent=2)+'\n')
        (RUN/'feedback_comparison.json').write_text(json.dumps(feedback_comparison(feedback), indent=2)+'\n')
        export_run(RUN)
    print(json.dumps(analysis, indent=2))
else:
    print('Definitions only; no actual phone episode, grading or algorithm advantage established')
